In [ ]:
# # UnicornForge AI — AMD Training Notebook
# Canonical training flow for `SuccessScoreMLP` on `global_startup_success_dataset.csv`.
# For reproducible training from CLI, use: `python train_model.py`


In [1]:
import torch
import pandas as pd

from ml.dataset import DEFAULT_DATASET_PATH, get_dataset_info
from ml.prompts import build_unicornforge_prompt, row_to_description
from ml.training import train_success_model

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


PyTorch: 2.12.1+cu130
CUDA available: True
Device: NVIDIA GeForce RTX 3060 Laptop GPU


In [ ]:
# ## 1. Load dataset


In [3]:
dataset_info = get_dataset_info()
print(dataset_info)

df = pd.read_csv(DEFAULT_DATASET_PATH)
print("Shape:", df.shape)
df.head()


{'loaded': True, 'path': '/home/piotrek/PycharmProjects/AMD Hackathon/UvicornForge-AI/backend/global_startup_success_dataset.csv', 'rows': 10000, 'columns': ['Startup Name', 'Founded Year', 'Country', 'Industry', 'Funding Stage', 'Total Funding ($)', 'Team Size', 'Monthly Recurring Revenue ($)', 'Valuation ($)', 'Success Score', 'Acquired?', 'Product Stage', 'Customer Base', 'Backend Tech Stack', 'Frontend Tech', 'Compute Platform', 'AMD Platform Used', 'Primary Model Used', 'Fireworks AI Credits Used ($, cumulative)', 'Social Media Followers']}
Shape: (10000, 20)


,Startup Name,Founded Year,Country,Industry,Funding Stage,Total Funding ($),Team Size,Monthly Recurring Revenue ($),Valuation ($),Success Score,Acquired?,Product Stage,Customer Base,Backend Tech Stack,Frontend Tech,Compute Platform,AMD Platform Used,Primary Model Used,"Fireworks AI Credits Used ($, cumulative)",Social Media Followers
0,Nectar Studio AI,2023,Canada,FinTech AI,Seed,9611.0,10,292.2,47900.0,5,No,Private Beta,2,"Node.js, Express",Flutter (Dart),Own AMD GPU cluster,ROCm on MI250 cluster,Mixtral 8x7B,0.00,1875
1,Zephyr Cloud AI,2025,Germany,Climate & Energy AI,Angel,1201.0,2,160.7,6800.0,3,No,MVP,150,"Java, Spring Boot + Kafka",Svelte,Fireworks AI API,—,MiniMax-M2.7,26.89,654
2,Amber Engine AI,2024,UK,Logistics & Supply Chain AI,Bootstrapped,15.0,2,0.7,1900.0,4,No,Private Beta,158,"Node.js, NestJS",React Native,Fireworks AI API,—,GPT-OSS-120B (high),26.11,494
3,FluxField,2024,USA,Gaming AI,Pre-Seed,1107.0,4,237.2,10300.0,2,No,MVP,25,"Java, Spring Boot",Vue,Own AMD GPU cluster,AMD Instinct MI250,Mixtral 8x7B,0.00,1752
4,AuroraWave.ai,2025,Germany,Logistics & Supply Chain AI,Pre-Seed,1194.0,5,71.1,13800.0,5,No,Private Beta,72,"Node.js, Express",React Native,Own AMD GPU cluster,ROCm on MI300X cluster,Mixtral 8x7B,0.00,670


In [ ]:
# ## 2. Build startup descriptions and prompts (used by brief generation)


In [7]:
df["startup_description"] = df.apply(row_to_description, axis=1)
df["uf_prompt"] = df["startup_description"].apply(build_unicornforge_prompt)
df[["Startup Name", "startup_description", "Success Score"]].head(5)


NameError: name 'employees' is not defined

In [ ]:
# ## 3. Train SuccessScoreMLP and export artifacts for backend inference


In [ ]:
result = train_success_model(epochs=20)
result


In [ ]:
# ## 4. Verify backend can load the exported model


In [ ]:
from ml.predictor import SuccessPredictor

predictor = SuccessPredictor()
print("Predictor ready:", predictor.ready)
print("Feature columns:", len(predictor.feature_columns))

sample = predictor.predict_from_payload(
    project_idea="AI tutor for university students",
    industry="EdTech",
    available_technologies="Python, AMD GPUs",
    available_time="48 hours",
)
print("Sample score:", sample.score if sample else None, sample.label if sample else None)


In [ ]:
# ## 5. Evaluate model quality (MAE / RMSE / R²)


In [ ]:
from ml.evaluation import evaluate_saved_model

metrics = evaluate_saved_model()
print("Validation metrics:", metrics["metrics"])
print("Sample predictions:", metrics["predictions"])


In [ ]:
# Optional: plot training loss curve if history is stored in metadata.


In [ ]:
import pickle
from pathlib import Path

metadata_path = Path("trained_models/startup_success_mlp/metadata.pkl")
if metadata_path.exists():
    with open(metadata_path, "rb") as handle:
        metadata = pickle.load(handle)
    history = metadata.get("history", [])
    if history:
        import matplotlib.pyplot as plt

        epochs = [item["epoch"] for item in history]
        train_mse = [item["train_mse"] for item in history]
        val_mse = [item["val_mse"] for item in history]
        plt.figure(figsize=(8, 4))
        plt.plot(epochs, train_mse, label="train MSE")
        plt.plot(epochs, val_mse, label="val MSE")
        plt.xlabel("Epoch")
        plt.ylabel("MSE")
        plt.title("SuccessScoreMLP training curve")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
    else:
        print("No training history in metadata. Retrain with: python train_model.py")
else:
    print("Metadata not found. Train the model first.")